# 05 — Debugging the Context Attribution Module

End-to-end debug harness for `context_attribution.context_cite`. This mimics a
**full test run** of the pipeline but is wired for inspection: every stage is
broken out so you can see the inputs and outputs as data flows through the module.

**What this notebook does and does not do:**

- **No backend API call.** The RAG response is loaded from the local
  `chat_output.txt` mock (a real captured backend payload), so `fetch_backend`
  is never invoked. This isolates the attribution module from the backend.
- **Still calls the Zingplay vLLM endpoint** for ContextCite. Every ablation
  scores the response with `prompt_logprobs=1` against the live LLM endpoint
  configured in `.env` (`OPENAI_BASE_URL` / `OPENAI_API_KEY`). That call path
  is exactly what we're debugging.

Pipeline mirrors `python -m context_attribution`:
`load mock → prepare_inputs → ContextCiter → get_attributions`.

## Parameters

All knobs for the run live here. Tweak and re-run top-to-bottom.

In [ ]:
# ============================ Parameters ============================

# Mock backend payload to debug against (no live backend call is made).
MOCK_PATH = 'chat_output.txt'
QUERY     = 'Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?'

# Context / response capacity caps (chars) applied by prepare_inputs.
# The module default (mini) is 2200 ctx / 500 resp, which keeps only 1 of 10
# chunks -> 1 source -> nothing to ablate. Raise MAX_CONTEXT_CHARS to admit more
# chunks (~6000 ≈ 8 chunks). Higher = more sources but a bigger prompt; if the
# endpoint returns 500 (prompt too long), lower it. None = no cap.
MAX_CONTEXT_CHARS  = 2200
MAX_RESPONSE_CHARS = 500

# ContextCite ablation settings.
NUM_ABLATIONS      = 32     # number of masked context variants scored per run
ABLATION_KEEP_PROB = 0.5    # probability each source is kept in a given ablation
BATCH_SIZE         = 1

# Logging: when True the module logs per-call token counts (and the size/HTTP
# status of any failing call) at INFO, so you can see how big each prompt is and
# exactly which call 500s.
VERBOSE_API_LOGGING = True

## Setup — import the module and inspect its config

In [ ]:
import sys, json, logging
from pathlib import Path
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Import the module under test. It self-configures from .env on import:
# the Zingplay LLM endpoint for prompt_logprobs and the mini-model char limits.
import context_attribution.context_cite as cc_module
from context_attribution.context_cite import (
    prepare_inputs,
    make_tokenizer,
    attribute_response,
    APIModel,
)
from context_cite import ContextCiter
from context_cite.context_citer import DEFAULT_PROMPT_TEMPLATE

# Apply the capacity caps from the Parameters cell (override the module defaults).
cc_module.MINI_MAX_CONTEXT_CHARS  = MAX_CONTEXT_CHARS
cc_module.MINI_MAX_RESPONSE_CHARS = MAX_RESPONSE_CHARS

# Logging: keep the root quiet (httpx/transformers stay at WARNING) but turn the
# module's per-call token logging up to INFO so it surfaces in the notebook.
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s', force=True)
logging.getLogger('context_attribution.context_cite').setLevel(
    logging.INFO if VERBOSE_API_LOGGING else logging.WARNING
)

print('Module under test   :', cc_module.__file__)
print('LLM endpoint (ZP)   :', cc_module.LLM_URL)
print('Model               :', cc_module.MODEL)
print('LLM API key set     :', bool(cc_module.API_KEY))
print('Max response chars  :', cc_module.MINI_MAX_RESPONSE_CHARS)
print('Max context chars   :', cc_module.MINI_MAX_CONTEXT_CHARS)
print('Verbose API logging :', VERBOSE_API_LOGGING)

# Sanity: we need the Zingplay endpoint for ContextCite, but NOT the backend.
assert cc_module.API_KEY,  'OPENAI_API_KEY missing — ContextCite needs the Zingplay endpoint'
assert cc_module.LLM_URL.startswith('http'), f'LLM_URL looks unset: {cc_module.LLM_URL!r}'
print('\nReady: Zingplay endpoint configured, backend will NOT be called.')

## Stage 1 — Load mock data (no backend call)

`chat_output.txt` is a captured backend payload (the same shape `fetch_backend`
returns). We load it straight from disk and pair it with the query that produced
it, so the backend is bypassed entirely.

In [ ]:
# MOCK_PATH and QUERY come from the Parameters cell.
# Load directly from file — fetch_backend() is deliberately NOT used here.
with open(MOCK_PATH) as f:
    data = json.load(f)

print('Loaded mock payload :', Path(MOCK_PATH).resolve())
print('Query               :', QUERY)
print('Top-level keys      :', list(data.keys()))
print()
print(f'answer              : {len(data["answer"])} chars')
print(f'knowledge_sources   : {len(data["knowledge_sources"])} chunks')
print(f'mode                : {data.get("mode")}')
print(f'usage               : {data.get("usage")}')
print()
print('--- raw answer ---')
print(data['answer'])
print()
print('--- knowledge source chunks ---')
for i, src in enumerate(data['knowledge_sources']):
    snip = ' '.join(src.get('content', '')[:100].split())
    print(f'  [{i}] score={src.get("score", 0):.3f}  {src.get("document_id", "<no doc>")}')
    print(f'       {snip}...')

## Stage 2 — `prepare_inputs`: build context + response

Applies the capacity caps from the Parameters cell and concatenates the surviving
chunks into the context string. The **before vs after** chunk count below is the
key signal: anything dropped here can never be credited by the attribution, and
if too few chunks survive there is nothing meaningful to ablate.

In [ ]:
context, response = prepare_inputs(data)

# --- Chunk count: before vs after preprocessing -------------------------
# 'before' = chunks the backend returned; 'after' = chunks that survive the
# MAX_CONTEXT_CHARS capacity cap. Instead of re-implementing prepare_inputs'
# truncation loop here (which can silently drift from the real one), derive the
# surviving count directly from prepare_inputs' returned `context`: each chunk
# begins with a markdown heading, so the number of heading blocks == the chunks
# that made it through. This uses the same regex the HeadingContextPartitioner
# applies in Stage 3, so n_after also matches cc.num_sources downstream.
import re
n_before = len(data['knowledge_sources'])
limit = cc_module.MINI_MAX_CONTEXT_CHARS
n_after = len(re.findall(r'(?m)^[ \t]*#', context)) if context else 0

print(f'Chunks before preprocessing : {n_before}')
print(f'Chunks after preprocessing  : {n_after}   (cap = {limit} context chars)')
print(f'Chunks dropped              : {n_before - n_after}')
print()
print('=== CONTEXT (', len(context), 'chars ) ===')
print(context)
print()
print('=== RESPONSE (', len(response), 'chars ) ===')
print(response)

if len(response) < len(data['answer']):
    print(f'\n[debug] response was truncated: {len(data["answer"])} -> {len(response)} chars')

## Stage 3 — Wire up `ContextCiter` with a heading-based partition

`make_tokenizer()` returns the GPT-2 shell tokenizer (ChatML template, used only
for structural bookkeeping). `APIModel` holds the fixed response so `generate()`
returns it without a live call.

**Source splitting (placeholder):** instead of ContextCite's default
sentence-level partitioner, we split the context at markdown headings (lines
starting with `#`) — i.e. roughly one source per chunk heading. This is a
deliberately coarse stand-in; a more custom splitter comes later. We pass it via
ContextCite's `partitioner=` hook so the rest of the pipeline is unchanged.

`cc.response` triggers `generate()` (no API call). `cc.sources` shows the
heading-level partition that the ablations will toggle.

In [ ]:
import re
import numpy as np
from typing import Optional
from context_cite.context_partitioner import BaseContextPartitioner


class HeadingContextPartitioner(BaseContextPartitioner):
    """Split context into sources at markdown headings (lines starting with '#').

    Placeholder, coarse splitter (~one source per chunk heading) used in place of
    ContextCite's sentence-level SimpleContextPartitioner. To be replaced by a
    more custom splitter later. Each source is a heading block: the heading line
    through to just before the next heading. get_context() reconstructs the
    context from any subset of sources for ablation.
    """

    def __init__(self, context: str) -> None:
        super().__init__(context)
        self._cache = {}

    def split_context(self) -> None:
        ctx = self.context
        starts = [m.start() for m in re.finditer(r'(?m)^[ \t]*#', ctx)]
        if not starts or starts[0] != 0:
            starts = [0] + starts          # leading text before the first heading = its own source
        parts, trailing = [], []
        for i, s in enumerate(starts):
            e = starts[i + 1] if i + 1 < len(starts) else len(ctx)
            block = ctx[s:e]
            body = block.rstrip()
            parts.append(body)
            trailing.append(block[len(body):])  # whitespace separating this block from the next
        self._cache['parts'] = parts
        self._cache['separators'] = [''] + trailing[:-1]

    @property
    def parts(self):
        if self._cache.get('parts') is None:
            self.split_context()
        return self._cache['parts']

    @property
    def separators(self):
        if self._cache.get('separators') is None:
            self.split_context()
        return self._cache['separators']

    @property
    def num_sources(self) -> int:
        return len(self.parts)

    def get_source(self, index: int) -> str:
        return self.parts[index]

    def get_context(self, mask: Optional[np.ndarray] = None):
        if mask is None:
            mask = np.ones(self.num_sources, dtype=bool)
        seps = np.array(self.separators, dtype=object)[mask]
        parts = np.array(self.parts, dtype=object)[mask]
        out = ''
        for i, (sep, part) in enumerate(zip(seps, parts)):
            if i > 0:
                out += sep
            out += part
        return out


tokenizer = make_tokenizer()
model = APIModel(response=response)
partitioner = HeadingContextPartitioner(context)

# Sanity: the heading split must reconstruct the original context exactly.
assert partitioner.get_context() == context, 'partitioner does not round-trip the context'

# NUM_ABLATIONS / ABLATION_KEEP_PROB / BATCH_SIZE come from the Parameters cell.
cc = ContextCiter(
    model=model,
    tokenizer=tokenizer,
    context=context,
    query=QUERY,
    num_ablations=NUM_ABLATIONS,
    ablation_keep_prob=ABLATION_KEEP_PROB,
    batch_size=BATCH_SIZE,
    partitioner=partitioner,   # heading-based split instead of sentence-level
)

print(f'num_sources (heading blocks): {cc.num_sources}')
print(f'num_ablations              : {NUM_ABLATIONS}')
print()
print('Sources (attribution units, split by # heading):')
for i, s in enumerate(cc.sources):
    print(f'  [{i}] {s[:90]}{"..." if len(s) > 90 else ""}')

print()
print('cc.response (via generate(), no API call):')
print(' ', cc.response[:200], '...' if len(cc.response) > 200 else '')

## Stage 4 — Debug a single Zingplay `prompt_logprobs` call

Before running all the ablations, verify the ContextCite scoring path works
against the live Zingplay endpoint with the **full (unmasked)** context. This is
the exact call `APIModel.__call__` makes per ablation; if this fails, the whole
run fails. We reconstruct the user content the way ContextCite does, then call
the module's `_api_response_token_logprobs` helper directly.

In [ ]:
# Reconstruct the unmasked user content exactly as ContextCite builds it.
# Use partitioner.get_context() (the same source-of-truth Stages 5/6 use) rather
# than the raw `context` variable, so this baseline matches lp_full in the delta
# test. The assert guards the round-trip: an unmasked get_context() must equal
# the original context, otherwise the baseline call here and lp_full diverge.
baseline_context = partitioner.get_context()
assert baseline_context == context, 'partitioner.get_context() != context — baseline would not match lp_full'

user_content = DEFAULT_PROMPT_TEMPLATE.format(context=baseline_context, query=QUERY)
print('user_content (', len(user_content), 'chars ) head:')
print(user_content[:300], '...\n')

# Live Zingplay call — this is the ContextCite scoring path under test.
api_tokens = cc_module._api_response_token_logprobs(user_content, response)

print(f'Zingplay returned {len(api_tokens)} scored response tokens')
total_lp = sum(lp for _, lp in api_tokens)
print(f'sum logprob = {total_lp:.3f}   mean = {total_lp / max(len(api_tokens), 1):.3f}')
print()
print('First 20 (token, logprob):')
for tok, lp in api_tokens[:20]:
    print(f'  {tok!r:20} {lp:8.3f}')

assert api_tokens, 'Zingplay returned no token logprobs — check endpoint / model / prompt size'
print('\nSingle-call scoring path OK.')

## Stage 4b — Find the context-size threshold (where the 500s start)

The endpoint returns 500 once the prompt exceeds the model's context window.
This pre-flight keeps the first *k* heading-sources for increasing *k* and calls
the endpoint each time, to pinpoint the size at which it begins to fail. The
module logs per-call token counts (INFO) alongside each row, and the HTTP status
of any failure. Use the breaking point to choose a safe `MAX_CONTEXT_CHARS` in
the Parameters cell.

In [ ]:
import requests

def probe(mask):
    """Call the endpoint with the masked context; return (ok, ctx_chars, prompt_chars, status)."""
    ctx = partitioner.get_context(mask)
    uc = DEFAULT_PROMPT_TEMPLATE.format(context=ctx, query=QUERY)
    try:
        cc_module._api_response_token_logprobs(uc, response)
        return True, len(ctx), len(uc), None
    except requests.HTTPError as e:
        return False, len(ctx), len(uc), e.response.status_code

n = cc.num_sources
print(f'{"kept":>4} {"ctx chars":>9} {"prompt chars":>12} {"result":>10}')
first_fail = None
for k in range(1, n + 1):
    mask = np.zeros(n, dtype=bool)
    mask[:k] = True
    ok, ctx_chars, prompt_chars, status = probe(mask)
    print(f'{k:>4} {ctx_chars:>9} {prompt_chars:>12} {"OK" if ok else f"FAIL {status}":>10}')
    if not ok and first_fail is None:
        first_fail = (k, ctx_chars)

if first_fail:
    k_fail, chars_fail = first_fail
    print(f'\nFirst failure at k={k_fail} sources kept (~{chars_fail} context chars).')
    print(f'Set MAX_CONTEXT_CHARS below {chars_fail} in the Parameters cell, then re-run.')
else:
    print('\nNo failures across the full sweep — every source count fits the window.')

## Stage 5 — Full attribution run (mimics the real test)

This is the full pipeline end-to-end: ContextCite masks random subsets of the
sources across `num_ablations` calls to Zingplay (`prompt_logprobs=1`), then fits
a Lasso to recover which sources drove the response.

We call `cc.get_attributions` directly (rather than the module's
`attribute_response`) so the run uses our heading-based partitioner from Stage 3
instead of the module's default sentence-level split.

`verbose=True` prints the per-ablation progress so you can watch the live calls.

In [ ]:
# Runs NUM_ABLATIONS live calls to the Zingplay endpoint.
result = cc.get_attributions(as_dataframe=True, verbose=True)
display(result)

## Stage 6 — Baseline validation: attribution + delta test

Two checks on the baseline (VNG leave-days) case:

1. **Well-formedness** — a row per source and finite scores.
2. **Delta test** — ablate the top-attributed source and measure the *actual*
   change in the response logprob against the full-context baseline (two extra
   live Zingplay calls). If the attribution is meaningful, removing the source
   that supports the answer should measurably lower the response's logprob
   (`delta > 0`). This grounds the Lasso scores in real model behavior rather
   than trusting them blindly.

In [ ]:
# --- 1. Well-formedness -------------------------------------------------
df = result.data if hasattr(result, 'data') else result
print('shape   :', df.shape)
print('columns :', list(df.columns))
assert df.shape[0] == cc.num_sources, 'row count should equal number of sources'

# Raw per-source scores (cached — no new API calls); index i = source i.
attrs = cc.get_attributions(as_dataframe=False, verbose=False)
assert np.isfinite(attrs).all(), 'non-finite attribution score'

top_idx = int(np.argmax(attrs))
print(f'\nTop source [{top_idx}] (score={attrs[top_idx]:.4f}):')
print(' ', cc.sources[top_idx][:120], '...')

# --- 2. Baseline delta test ---------------------------------------------
# Ablate the top-attributed source and measure the ACTUAL change in response
# logprob vs the full context. Removing the source that supports the answer
# should lower the response logprob (delta > 0).
def response_logprob(mask):
    ctx = partitioner.get_context(mask)
    uc = DEFAULT_PROMPT_TEMPLATE.format(context=ctx, query=QUERY)
    toks = cc_module._api_response_token_logprobs(uc, response)
    return sum(lp for _, lp in toks)

mask_full = np.ones(cc.num_sources, dtype=bool)
mask_drop = mask_full.copy()
mask_drop[top_idx] = False

lp_full = response_logprob(mask_full)
lp_drop = response_logprob(mask_drop)
delta = lp_full - lp_drop

print('\n--- Baseline delta test (top source ablated) ---')
print(f'logprob, full context        : {lp_full:8.3f}')
print(f'logprob, top source removed  : {lp_drop:8.3f}')
print(f'delta (full - ablated)       : {delta:8.3f}')

assert delta > 0, f'ablating the top source did NOT lower the response logprob (delta={delta:.3f})'
print('\nPASS: top-attributed source measurably supports the response. Full debug run complete.')